# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32904\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra columns for interaction features

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

In [20]:
df['age_hypertension'] = df['age'] * df['hypertension']

In [21]:
df['age_heart_disease'] = df['age'] * df['heart_disease']

In [22]:
df['age_bmi'] = df['age'] * df['bmi']

In [23]:
df['age_avg_glucose_level'] = df['age'] * df['avg_glucose_level']

In [24]:
df['bmi_avg_glucose_level'] = df['bmi'] * df['avg_glucose_level']

In [25]:
df['avg_glucose_level_hypertension'] = df['avg_glucose_level'] * df['hypertension']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [26]:
df['obese'] = df['bmi'] >= 30

In [27]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

In [28]:
df['obese_high_avg_glucose_level'] = df['obese'] | df['high_avg_glucose_level']

### Creating column representing high age

In [29]:
df['old'] = df['age'] >= 65

### Converting boolean columns to numerical

In [30]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Accounting for multiple risks

In [31]:
df['risk_count'] = df['obese'] + df['high_avg_glucose_level'] + df['old'] + df['hypertension'] + df['heart_disease']

### Reformatting column names

In [32]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [33]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [34]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [35]:
df['log_bmi'] = np.log(df['bmi'])

In [36]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [37]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,age_hypertension,age_heart_disease,age_bmi,age_avg_glucose_level,bmi_avg_glucose_level,avg_glucose_level_hypertension,obese,high_avg_glucose_level,obese_high_avg_glucose_level,old,risk_count,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,0.0,67.0,2452.2,15322.23,8370.054,0.00,1,1,1,1,4,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0.0,0.0,1714.1,12334.81,5682.101,0.00,0,1,1,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,0.0,80.0,2600.0,8473.60,3442.400,0.00,1,0,1,1,3,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,0.0,0.0,1685.6,8390.27,5890.312,0.00,1,1,1,0,2,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,79.0,0.0,1896.0,13755.48,4178.880,174.12,0,1,1,1,3,3.178054,5.159745


In [38]:
df.shape

(5110, 36)

## Model Building

### Splitting to training, validation, and testing sets

In [39]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [40]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_32904\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### Create base random forest classifier model

In [42]:
base_classifier = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

### Create grid for hyperparameter tuning

In [43]:
param_grid_xgb = {
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'scale_pos_weight': [1, 3, 5, 10, 15, 20]
}

In [44]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [45]:
grid_xgb = GridSearchCV(
    estimator=base_classifier,
    param_grid=param_grid_xgb,
    scoring=f2_scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

### Fitting base classifier on unsampled data

In [46]:
base_classifier.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [47]:
importances = base_classifier.feature_importances_

In [48]:
feature_imp_df = pd.DataFrame({'Feature':[col for col in X_train], 'Importance': importances}).sort_values(
    'Importance', ascending=False)

In [49]:
print(feature_imp_df)

                           Feature  Importance
0                              age    0.148926
3                     ever_married    0.083450
23                         age_bmi    0.058028
19           smoking_status_smokes    0.053031
21                age_hypertension    0.048881
5                              bmi    0.043420
11              work_type_govt_job    0.039279
18     smoking_status_never_smoked    0.037942
31                      risk_count    0.036048
4                avg_glucose_level    0.034835
26  avg_glucose_level_hypertension    0.033516
10   hypertension_or_heart_disease    0.032696
20                age_ever_married    0.032403
25           bmi_avg_glucose_level    0.031465
16          smoking_status_unknown    0.030754
22               age_heart_disease    0.028763
8                            smoke    0.028084
7                            urban    0.027726
6                             male    0.027300
2                    heart_disease    0.027104
14         wo

In [50]:
# retained_columns = ['age', 'ever_married', 'age_bmi', 'smoking_status_smokes', 'age_hypertension', 'bmi', 'work_type_govt_job', 'smoking_status_never_smoked']

In [51]:
# retained_columns = ['age', 'ever_married', 'age_bmi', 'smoking_status_smokes', 'age_hypertension', 'bmi', 'work_type_govt_job', 'smoking_status_never_smoked', 'risk_count']

In [52]:
# retained_columns = ['age', 'ever_married', 'age_bmi', 'smoking_status_smokes', 'age_hypertension', 'bmi', 'work_type_govt_job', 'smoking_status_never_smoked', 'risk_count', 'avg_glucose_level']

In [53]:
# retained_columns = ['age', 'smoking_status_unknown', 'ever_married', 'smoking_status_smokes', 'smoking_status_never_smoked', 'age_heart_disease', 'bmi', 'age_hypertension']

In [54]:
# retained_columns = ['age', 'smoking_status_unknown', 'ever_married', 'smoking_status_smokes', 'smoking_status_never_smoked', 'age_heart_disease', 'bmi', 'age_hypertension', 'work_type_private', 'age_ever_married', 'work_type_self_employed', 'urban']

In [55]:
retained_columns = ['age', 'smoking_status_unknown', 'ever_married', 'smoking_status_smokes', 'smoking_status_never_smoked', 'age_heart_disease', 'bmi', 'age_hypertension', 'work_type_private', 'age_ever_married']

In [56]:
# retained_columns = ['age', 'smoking_status_unknown', 'ever_married', 'smoking_status_smokes', 'smoking_status_never_smoked', 'age_heart_disease', 'bmi', 'age_hypertension', 'work_type_private']

In [57]:
X_train_retained = X_train[retained_columns]

In [58]:
X_val_retained = X_val[retained_columns]

### Create optimizing function for hyperparameter tuning with Optuna

In [59]:
MIN_PRECISION = 0.2

In [60]:
def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.4, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 20),
        'tree_method': trial.suggest_categorical('tree_method', ['hist', 'approx']), 
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),  
        'max_leaves': trial.suggest_int('max_leaves', 0, 256),
        'min_split_loss': trial.suggest_float('min_split_loss', 0, 5),  
        'max_bin': trial.suggest_int('max_bin', 128, 512),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = xgb.XGBClassifier(**params)
    
    model.fit(X_train_retained, y_train)
    
    y_pred_val = model.predict(X_val_retained)
    
    recall = recall_score(y_val, y_pred_val)
    precision = precision_score(y_val, y_pred_val)
    f2_score = fbeta_score(y_val, y_pred_val, beta=2)
    
    if precision < MIN_PRECISION:
        return 0
    return recall + (precision * 0.01)
    # return f2_score + recall * 0.001 + precision * 0.00001

In [61]:
study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=500, show_progress_bar=True)

[I 2026-02-02 21:54:49,367] A new study created in memory with name: no-name-acb2574c-67ab-477c-a0fb-a6b47e828fb2


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-02-02 21:54:49,781] Trial 0 finished with value: 0.0 and parameters: {'max_depth': 7, 'learning_rate': 0.24517932047070642, 'n_estimators': 380, 'min_child_weight': 6, 'max_delta_step': 1, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.4348501673009197, 'colsample_bylevel': 0.9197056874649611, 'colsample_bynode': 0.7606690070459252, 'gamma': 3.540362888980227, 'reg_alpha': 0.20584494295802447, 'reg_lambda': 9.699098521619943, 'scale_pos_weight': 16.816410175208013, 'tree_method': 'hist', 'grow_policy': 'lossguide', 'max_leaves': 134, 'min_split_loss': 2.1597250932105787, 'max_bin': 240}. Best is trial 0 with value: 0.0.


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-02-02 21:54:50,210] Trial 1 finished with value: 0.0 and parameters: {'max_depth': 10, 'learning_rate': 0.008851384099881301, 'n_estimators': 181, 'min_child_weight': 4, 'max_delta_step': 5, 'subsample': 0.8925879806965068, 'colsample_bytree': 0.5198042692950159, 'colsample_bylevel': 0.708540663048167, 'colsample_bynode': 0.7554487413172255, 'gamma': 0.23225206359998862, 'reg_alpha': 6.075448519014383, 'reg_lambda': 1.7052412368729153, 'scale_pos_weight': 2.235980266720311, 'tree_method': 'approx', 'grow_policy': 'depthwise', 'max_leaves': 25, 'min_split_loss': 3.4211651325607844, 'max_bin': 297}. Best is trial 0 with value: 0.0.
[I 2026-02-02 21:54:50,300] Trial 2 finished with value: 0.0 and parameters: {'max_depth': 4, 'learning_rate': 0.03797252233376349, 'n_estimators': 65, 'min_child_weight': 10, 'max_delta_step': 2, 'subsample': 0.831261142176991, 'colsample_bytree': 0.5870266456536466, 'colsample_bylevel': 0.7120408127066865, 'colsample_bynode': 0.7280261676059678, 'gam

...

[I 2026-02-02 21:55:07,870] Trial 46 finished with value: 0.0 and parameters: {'max_depth': 15, 'learning_rate': 0.006022414709183384, 'n_estimators': 212, 'min_child_weight': 6, 'max_delta_step': 10, 'subsample': 0.5751974652226473, 'colsample_bytree': 0.9587166215113213, 'colsample_bylevel': 0.9703076089888533, 'colsample_bynode': 0.6135154743754306, 'gamma': 3.14998367245953, 'reg_alpha': 6.203825128838996, 'reg_lambda': 3.0019052746564654, 'scale_pos_weight': 2.56647088112381, 'tree_method': 'hist', 'grow_policy': 'lossguide', 'max_leaves': 185, 'min_split_loss': 3.2996681025145485, 'max_bin': 415}. Best is trial 29 with value: 0.4614594594594595.
[I 2026-02-02 21:55:08,074] Trial 47 finished with value: 0.08380835380835382 and parameters: {'max_depth': 12, 'learning_rate': 0.016829489855796274, 'n_estimators': 72, 'min_child_weight': 9, 'max_delta_step': 8, 'subsample': 0.5106777956746407, 'colsample_bytree': 0.8264667475027827, 'colsample_bylevel': 0.9654743404399069, 'colsample_

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-02-02 21:55:09,807] Trial 50 finished with value: 0.0 and parameters: {'max_depth': 14, 'learning_rate': 0.09949450170160976, 'n_estimators': 105, 'min_child_weight': 4, 'max_delta_step': 1, 'subsample': 0.5818553915742474, 'colsample_bytree': 0.6045747934127993, 'colsample_bylevel': 0.7947937037624382, 'colsample_bynode': 0.5768561104829175, 'gamma': 4.340559153263046, 'reg_alpha': 4.341356160300686, 'reg_lambda': 1.2633069372328778, 'scale_pos_weight': 1.3258724273076314, 'tree_method': 'approx', 'grow_policy': 'lossguide', 'max_leaves': 206, 'min_split_loss': 1.7923453148073931, 'max_bin': 395}. Best is trial 29 with value: 0.4614594594594595.
[I 2026-02-02 21:55:10,352] Trial 51 finished with value: 0.24597051597051597 and parameters: {'max_depth': 13, 'learning_rate': 0.007014726911131919, 'n_estimators': 174, 'min_child_weight': 8, 'max_delta_step': 9, 'subsample': 0.6099926201252741, 'colsample_bytree': 0.8838627899277368, 'colsample_bylevel': 0.8905898338135135, 'colsam

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-02-02 21:57:57,949] Trial 382 finished with value: 0.0 and parameters: {'max_depth': 6, 'learning_rate': 0.06817487666868674, 'n_estimators': 187, 'min_child_weight': 7, 'max_delta_step': 10, 'subsample': 0.9057114814072367, 'colsample_bytree': 0.454831266214088, 'colsample_bylevel': 0.8303256653294292, 'colsample_bynode': 0.5710286931575097, 'gamma': 0.5142944707499623, 'reg_alpha': 7.927690814572234, 'reg_lambda': 8.089403546311866, 'scale_pos_weight': 1.630866809339107, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'max_leaves': 101, 'min_split_loss': 1.2251099830030976, 'max_bin': 167}. Best is trial 314 with value: 0.7048165238409141.
[I 2026-02-02 21:57:58,366] Trial 383 finished with value: 0.5965945945945946 and parameters: {'max_depth': 8, 'learning_rate': 0.06291887359703566, 'n_estimators': 198, 'min_child_weight': 7, 'max_delta_step': 9, 'subsample': 0.9113145558381401, 'colsample_bytree': 0.44256005127988535, 'colsample_bylevel': 0.8559065153707329, 'colsample

In [62]:
# grid_xgb.fit(X_train_retained, y_train)

In [63]:
# best_params_xgb = grid_xgb.best_params_

In [64]:
best_params_xgb = study_xgb.best_params

In [65]:
print(best_params_xgb)

{'max_depth': 8, 'learning_rate': 0.05455471168883755, 'n_estimators': 186, 'min_child_weight': 7, 'max_delta_step': 9, 'subsample': 0.9150538744186808, 'colsample_bytree': 0.4776895536294967, 'colsample_bylevel': 0.8074122734272388, 'colsample_bynode': 0.5731967866429041, 'gamma': 0.17409349437318175, 'reg_alpha': 0.017374679636583235, 'reg_lambda': 7.834188155939701, 'scale_pos_weight': 10.611449136206812, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'max_leaves': 105, 'min_split_loss': 1.4238466980269138, 'max_bin': 170}


In [66]:
final_classifier = xgb.XGBClassifier(**best_params_xgb, random_state=42, eval_metric='logloss')

In [67]:
final_classifier.fit(X_train_retained, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.8074122734272388,
              colsample_bynode=0.5731967866429041,
              colsample_bytree=0.4776895536294967, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=0.17409349437318175, grow_policy='depthwise',
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05455471168883755, max_bin=170,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=9,
              max_depth=8, max_leaves=105, min_child_weight=7,
              min_split_loss=1.4238466980269138, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=186,
              n_jobs=None, ...)

In [68]:
y_pred_val = final_classifier.predict(X_val_retained)

In [69]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [70]:
print(confusion_matrix(y_val, y_pred_val))

[[632  97]
 [ 11  26]]


In [71]:
print(accuracy_score(y_val, y_pred_val))

0.8590078328981723


In [72]:
print(precision_score(y_val, y_pred_val))

0.21138211382113822


In [73]:
print(recall_score(y_val, y_pred_val))

0.7027027027027027


In [74]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.4797047970479705


### Threshold tuning

In [75]:
best_threshold = 0.5

In [76]:
best_precision_score = 0

In [77]:
best_recall_score = 0

In [78]:
best_f2_score = 0

In [79]:
for threshold in np.arange(0.1, 0.95, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has F2 score {f2_score}')
    
    if precision >= 0.15:
        current_recall_score = recall
        if current_recall_score > best_recall_score:
            best_recall_score = current_recall_score
            best_precision_score = precision
            best_threshold = threshold
        elif current_recall_score == best_recall_score and precision > best_precision_score:
            best_precision_score = precision
            best_threshold = threshold
        
        # current_f2_score = f2_score
        # if current_f2_score > best_f2_score:
        #     best_f2_score = current_f2_score
        #     best_threshold = threshold 

Threshold 0.1 has F2 score 0.33081285444234404
Threshold 0.15000000000000002 has F2 score 0.3340080971659919
Threshold 0.20000000000000004 has F2 score 0.35181236673773986
Threshold 0.25000000000000006 has F2 score 0.36666666666666664
Threshold 0.30000000000000004 has F2 score 0.37349397590361444
Threshold 0.3500000000000001 has F2 score 0.4068241469816273
Threshold 0.40000000000000013 has F2 score 0.42028985507246375
Threshold 0.45000000000000007 has F2 score 0.4368932038834951
Threshold 0.5000000000000001 has F2 score 0.4797047970479705
Threshold 0.5500000000000002 has F2 score 0.371900826446281
Threshold 0.6000000000000002 has F2 score 0.365296803652968
Threshold 0.6500000000000001 has F2 score 0.28350515463917525
Threshold 0.7000000000000002 has F2 score 0.25
Threshold 0.7500000000000002 has F2 score 0.12269938650306748
Threshold 0.8000000000000002 has F2 score 0.06578947368421052
Threshold 0.8500000000000002 has F2 score 0.03333333333333333
Threshold 0.9000000000000002 has F2 scor

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [80]:
print(best_threshold)

0.45000000000000007


In [81]:
print(best_f2_score)

0


### Retrain on training + validation set

In [82]:
X_train_full = pd.concat([X_train_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train, y_val], axis=0, ignore_index=True)

In [83]:
final_classifier.fit(X_train_full, y_train_full)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.8074122734272388,
              colsample_bynode=0.5731967866429041,
              colsample_bytree=0.4776895536294967, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=0.17409349437318175, grow_policy='depthwise',
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05455471168883755, max_bin=170,
              max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=9,
              max_depth=8, max_leaves=105, min_child_weight=7,
              min_split_loss=1.4238466980269138, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=186,
              n_jobs=None, ...)

### Evaluate on test set

In [84]:
X_test_retained = X_test[retained_columns]

In [85]:
y_pred_test = final_classifier.predict(X_test_retained)

In [86]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [87]:
print(confusion_matrix(y_test, y_pred_test))

[[621 108]
 [ 11  27]]


In [88]:
print(accuracy_score(y_test, y_pred_test))

0.8448500651890483


In [89]:
print(precision_score(y_test, y_pred_test))

0.2


In [90]:
print(recall_score(y_test, y_pred_test))

0.7105263157894737


In [91]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.47038327526132406


### Test with best threshold 0.55

In [92]:
y_pred_test_threshold = (y_proba_test >= best_threshold).astype(int)

In [93]:
print(confusion_matrix(y_test, y_pred_test_threshold))

[[595 134]
 [ 10  28]]


In [94]:
print(accuracy_score(y_test, y_pred_test_threshold))

0.8122555410691004


In [95]:
print(precision_score(y_test, y_pred_test_threshold))

0.1728395061728395


In [96]:
print(recall_score(y_test, y_pred_test_threshold))

0.7368421052631579


In [97]:
print(fbeta_score(y_test, y_pred_test_threshold, beta=2))

0.445859872611465
